# Evaluación del extractor (heurística de asociación)

`train.ipynb` evalúa el modelo LayoutLMv3 (clasificación de tokens) sobre el split de `data/labeled/`.
Este notebook evalúa la capa que se construye encima: `KeyValueExtractor.apply_heuristic`, que
resuelve pares clave-valor y filas de tabla a partir de las labels — y lo hace sobre un set de
evaluación **separado y held-out**: `evaluate/jsons/` (labels reales) + `evaluate/pdfs/` (PDFs
correspondientes), documentos que nunca formaron parte del pool de entrenamiento (`data/labeled/`).

Metodología, en 3 partes:

1. **Heurística en aislamiento** — corremos `apply_heuristic` directamente sobre las labels *reales*
   (de `evaluate/jsons/`), sin pasar por el modelo. Esto muestra qué tan buena es la heurística de
   asociación cuando la clasificación de tokens es perfecta (aísla el error de asociación del error
   del modelo).
2. **Pipeline completo end-to-end** — corremos `extractor.predict(pdf)` (modelo + heurística) sobre
   los PDFs de `evaluate/pdfs/`, y comparamos contra la salida "gold" del punto 1.
3. **Calibración de `is_reliable`** — cruzamos la bandera de confiabilidad contra si el valor
   extraído fue correcto o no, para validar cuantitativamente la contribución de tesis #3.

**Limitación conocida:** no existe un "gold" de pares clave-valor ya resueltos (el JSON etiquetado es
a nivel de token/entidad, no de pares). Usamos como referencia la salida de `apply_heuristic` sobre
las labels reales — es decir, la propia heurística aplicada a clasificación perfecta. Esto es válido
para aislar error-de-modelo vs error-de-heurística, pero **no** valida por sí solo si la heurística
elige el pareo "humanamente correcto" cuando hay ambigüedad real (p. ej. dos `FIELD_KEY_ID` compitiendo
por el mismo valor). Para eso se necesitaría un gold set curado a mano sobre una muestra pequeña —
queda fuera de este notebook.

In [40]:
import os
import sys
from pathlib import Path
ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [41]:
import json
from collections import defaultdict

import pandas as pd

from extract.key_value_extractor import KeyValueExtractor

LABELED_DIR = Path("evaluate/jsons")
PDF_DIR = Path("evaluate/pdfs")
TRAIN_LABELED_DIR = Path("data/labeled")  # pool usado en train.ipynb, solo para el chequeo de fuga

extractor = KeyValueExtractor()

c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 1. Heurística en aislamiento (labels reales -> `apply_heuristic`)

In [42]:
def load_gold_results(json_path: Path):
    """Convierte un JSON de data/labeled (lista plana de entidades) al formato
    que espera apply_heuristic: una lista de páginas con words/boxes/labels."""
    with open(json_path, encoding="utf-8") as f:
        entities = json.load(f)

    by_page = defaultdict(list)
    for e in entities:
        by_page[e["page"]].append(e)

    results = []
    for page, page_entities in sorted(by_page.items()):
        results.append({
            "page": page,
            "words": [e["text"] for e in page_entities],
            "boxes": [e["normalized_bbox"] for e in page_entities],
            "labels": [e["label"] for e in page_entities],
        })
    return results

In [43]:
labeled_files = sorted(LABELED_DIR.glob("*.json"))
gold_outputs = {}
for path in labeled_files:
    doc_id = path.stem
    gold_results = load_gold_results(path)
    gold_outputs[doc_id] = extractor.apply_heuristic(gold_results)

print(f"Documentos de evaluación: {len(gold_outputs)}")

Documentos de evaluación: 5


In [44]:
# Chequeo de fuga: ninguno de estos documentos debería estar en el pool de train.ipynb
eval_ids = set(gold_outputs)
train_pool_ids = {p.stem for p in TRAIN_LABELED_DIR.glob("*.json")}
leaked = eval_ids & train_pool_ids

assert not leaked, f"Fuga de datos: estos IDs de evaluate/ tambien estan en data/labeled/: {leaked}"
print("OK: ningun documento de evaluate/ esta en el pool de entrenamiento (data/labeled/).")

OK: ningun documento de evaluate/ esta en el pool de entrenamiento (data/labeled/).


## 2. Pipeline completo (modelo + heurística) sobre `evaluate/pdfs/`

In [45]:
available_pdf_ids = {p.stem for p in PDF_DIR.glob("*.pdf")}
common_ids = sorted(available_pdf_ids & set(gold_outputs))

missing_pdf = set(gold_outputs) - available_pdf_ids
if missing_pdf:
    print(f"Aviso: {len(missing_pdf)} documentos etiquetados sin PDF en evaluate/pdfs/: {missing_pdf}")


In [46]:
pipeline_outputs = {}
for doc_id in common_ids:
    pdf_path = PDF_DIR / f"{doc_id}.pdf"
    pipeline_outputs[doc_id] = extractor.predict(pdf_path)



c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


## 3. Comparación campo a campo (form)

In [47]:
def normalize(value):
    return value.strip().upper() if isinstance(value, str) else value


def match_form(gold_form, pipeline_form):
    gold_by_field = defaultdict(list)
    for entry in gold_form:
        gold_by_field[entry["field"]].append(entry)

    pipeline_by_field = defaultdict(list)
    for entry in pipeline_form:
        pipeline_by_field[entry["field"]].append(entry)

    records = []
    fields = sorted(set(gold_by_field) | set(pipeline_by_field))
    for field in fields:
        golds = gold_by_field.get(field, [])
        preds = pipeline_by_field.get(field, [])

        # Se evaluan todos los valores
        for i in range(max(len(golds), len(preds))):
            gold_entry = golds[i] if i < len(golds) else None
            pred_entry = preds[i] if i < len(preds) else None

            if gold_entry and pred_entry:
                correct = normalize(gold_entry["value"]) == normalize(pred_entry["value"])
            else:
                correct = None
                
            records.append({
                "field": field,
                "gold_value": gold_entry["value"] if gold_entry else None,
                "pipeline_value": pred_entry["value"] if pred_entry else None,
                "is_reliable": pred_entry["is_reliable"] if pred_entry else None,
                "correct": correct
            })
    return records

In [48]:
form_records = []
for doc_id in common_ids:
    for record in match_form(gold_outputs[doc_id]["form"], pipeline_outputs[doc_id]["form"]):
        record["doc_id"] = doc_id
        form_records.append(record)

form_df = pd.DataFrame(form_records)
form_df.head(10)

,field,gold_value,pipeline_value,is_reliable,correct,doc_id
0,AMBIENTE:,PRODUCCIÓN,PRODUCCIÓN,True,True,1106202601139174848500120080200000487350004873513
1,Agente de Retención Resolución No.,1,1,True,True,1106202601139174848500120080200000487350004873513
2,Contribuyente Especial,0011,0011,True,True,1106202601139174848500120080200000487350004873513
3,Direccion:,MONTECRISTI,MONTECRISTI,True,True,1106202601139174848500120080200000487350004873513
4,Dirección Matriz:,KM 3.5 VIA PORTOVIEJO-CRUCITA,KM 3.5 VIA PORTOVIEJO-CRUCITA,True,True,1106202601139174848500120080200000487350004873513
5,Dirección Sucursal:,PANAMERICANA S/N Y BOLIVAR,PANAMERICANA S/N Y BOLIVAR,True,True,1106202601139174848500120080200000487350004873513
6,EMISIÓN:,NORMAL,NORMAL,True,True,1106202601139174848500120080200000487350004873513
7,FECHA Y HORA DE AUTORIZACIÓN:,11/06/2026 16:51:06,11/06/2026 16:51:06,True,True,1106202601139174848500120080200000487350004873513
8,Fecha,11/06/2026,11/06/2026,True,True,1106202601139174848500120080200000487350004873513
9,Forma de pago,20 - OTROS CON UTILIZACION DEL SISTEMA FINANCIERO,20 - OTROS CON UTILIZACION DEL SISTEMA FINANCIERO,True,True,1106202601139174848500120080200000487350004873513


## 4. Métricas de extracción (form)

In [49]:
def precision_recall_f1(df):
    tp = int((df["correct"] == True).sum())
    fp = int(df["pipeline_value"].notna().sum()) - tp
    fn = int(df["gold_value"].notna().sum()) - tp
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else float("nan")
    )
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


form_metrics = precision_recall_f1(form_df)
form_metrics

{'tp': 137,
 'fp': 0,
 'fn': 6,
 'precision': 1.0,
 'recall': 0.958041958041958,
 'f1': 0.9785714285714286}

## 5. Calibración de `is_reliable` (validación contribución #3)

In [50]:
def calibration_report(df):
    evaluated = df.dropna(subset=["correct", "is_reliable"])
    crosstab = pd.crosstab(evaluated["is_reliable"], evaluated["correct"])

    acc_reliable = evaluated.loc[evaluated["is_reliable"] == True, "correct"].mean()

    return {
        "crosstab": crosstab,
        "accuracy_if_reliable": acc_reliable,
    }


form_calibration = calibration_report(form_df)
print(form_calibration["crosstab"])
print()
for k, v in form_calibration.items():
    if k != "crosstab":
        print(f"{k}: {v}")

correct      True
is_reliable      
True          137

accuracy_if_reliable: 1.0


## 6. Comparación de tablas (headers/rows)

Alineamos tablas y filas por índice (misma tabla lógica y mismo orden de fila entre la corrida gold
y la corrida del pipeline). Es una aproximación razonable porque ambas corridas usan la misma lógica
de agrupamiento por posición (`_build_single_table`), pero se puede romper si el modelo pierde o
inventa una fila completa — en ese caso las filas subsiguientes quedan desalineadas. Vale la pena
inspeccionar manualmente los casos con mayor desacuerdo antes de citar el número final.

In [51]:
def match_tables(gold_tables, pipeline_tables):
    records = []
    for i in range(max(len(gold_tables), len(pipeline_tables))):
        gold_table = gold_tables[i] if i < len(gold_tables) else {"headers": [], "rows": []}
        pipeline_table = pipeline_tables[i] if i < len(pipeline_tables) else {"headers": [], "rows": []}

        headers = pipeline_table["headers"] or gold_table["headers"]
        n_rows = max(len(gold_table["rows"]), len(pipeline_table["rows"]))

        for j in range(n_rows):
            gold_row = gold_table["rows"][j] if j < len(gold_table["rows"]) else None
            pred_row = pipeline_table["rows"][j] if j < len(pipeline_table["rows"]) else None

            for k, column in enumerate(headers):
                gold_cell = gold_row[k] if gold_row and k < len(gold_row) else None
                pred_cell = pred_row[k] if pred_row and k < len(pred_row) else None

                if gold_cell and pred_cell:
                    correct = normalize(gold_cell["value"]) == normalize(pred_cell["value"])
                else:
                    correct = None

                records.append({
                    "table": i,
                    "row": j,
                    "column": column,
                    "gold_value": gold_cell["value"] if gold_cell else None,
                    "pipeline_value": pred_cell["value"] if pred_cell else None,
                    "is_reliable": pred_cell["is_reliable"] if pred_cell else None,
                    "correct": correct,
                })
    return records

In [52]:
table_records = []
for doc_id in common_ids:
    for record in match_tables(gold_outputs[doc_id]["tables"], pipeline_outputs[doc_id]["tables"]):
        record["doc_id"] = doc_id
        table_records.append(record)

table_df = pd.DataFrame(table_records)
table_df.head(10)

,table,row,column,gold_value,pipeline_value,is_reliable,correct,doc_id
0,0,0,Cod. Principal,491,491,True,True,1106202601139174848500120080200000487350004873513
1,0,0,Cod. Auxiliar,NaN,NaN,None,None,1106202601139174848500120080200000487350004873513
2,0,0,Cantidad,1.00,1.00,True,True,1106202601139174848500120080200000487350004873513
3,0,0,Descripción,AVPF LUDAMA BOTOX VITRIFICx300GR,AVPF LUDAMA BOTOX VITRIFICx300GR,True,True,1106202601139174848500120080200000487350004873513
4,0,0,Detalle Adicional,NaN,NaN,None,None,1106202601139174848500120080200000487350004873513
5,0,0,Precio Unitario,13.9594,13.9594,True,True,1106202601139174848500120080200000487350004873513
6,0,0,Subsidio,0.00,0.00,True,True,1106202601139174848500120080200000487350004873513
7,0,0,Precio sin Subsidio,0.00,0.00,True,True,1106202601139174848500120080200000487350004873513
8,0,0,Descuento,0.00,0.00,True,True,1106202601139174848500120080200000487350004873513
9,0,0,Precio Total,13.96,13.96,True,True,1106202601139174848500120080200000487350004873513


## 7. Métricas de tablas

In [53]:
table_metrics = precision_recall_f1(table_df)
table_calibration = calibration_report(table_df)

print(table_metrics)
print()
print(table_calibration["crosstab"])
print()
for k, v in table_calibration.items():
    if k != "crosstab":
        print(f"{k}: {v}")

{'tp': 146, 'fp': 0, 'fn': 43, 'precision': 1.0, 'recall': 0.7724867724867724, 'f1': 0.871641791044776}

correct      True
is_reliable      
True          146

accuracy_if_reliable: 1.0


## 8. Resumen final

In [54]:
summary = pd.DataFrame([
    {"scope": "form", **form_metrics,
     "accuracy_if_reliable": form_calibration["accuracy_if_reliable"]},
    {"scope": "tables", **table_metrics,
     "accuracy_if_reliable": table_calibration["accuracy_if_reliable"]},
])
summary

,scope,tp,fp,fn,precision,recall,f1,accuracy_if_reliable
0,form,137,0,6,1.0,0.958042,0.978571,1.0
1,tables,146,0,43,1.0,0.772487,0.871642,1.0
